In [40]:
import os
!pip install -U langchain-google-genai

os.environ['GEMINI_API_KEY'] = "AQ.Ab8RN6J-YjTRgGVL9JJo45b6LCAggZWTrMdpGA2a9Xjnuk8gBQ"

In [41]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

In [56]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> dict:
  """
  This function fetches the currency factor between a given base currency and a target currency
  """
  import requests
  url = f'https://v6.exchangerate-api.com/v6/0b99f2baa745d80f2b7f6a62/pair/{base_currency}/{target_currency}'
  response = requests.get(url)
  return response.json()

@tool
def convert(base_value: float, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  Given a currency conversion rate, this function calculates the target currency value from a given base currency value.
  """
  return base_value * conversion_rate

In [57]:
convert.args

{'base_value': {'title': 'Base Value', 'type': 'number'}}

In [63]:
get_conversion_factor.invoke({
    'base_currency': 'USD',
    'target_currency': 'INR'
})


{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1789171201,
 'time_last_update_utc': 'Sat, 12 Sep 2026 00:00:01 +0000',
 'time_next_update_unix': 1789257601,
 'time_next_update_utc': 'Sun, 13 Sep 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.6045}

In [64]:
convert.invoke({'base_value': 10, 'conversion_rate': 95.5306})


955.306

In [65]:
llm = ChatGoogleGenerativeAI

In [66]:
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [67]:
messages =[HumanMessage('what is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr')]

In [68]:
messages

[HumanMessage(content='what is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [69]:
ai_message=llm_with_tools.invoke(messages)

In [70]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'target_currency': 'INR', 'base_currency': 'USD'},
  'id': 'call_1318776',
  'type': 'tool_call'}]

In [80]:
import json
for tool_call in ai_message.tool_calls:
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1= get_conversion_factor.invoke(input=tool_call['args'])
    print(tool_message1)

    print(tool_message1['conversion_rate'])

    messages.append(tool_message1)

  if tool_call['name']=='convert':

    tool_call['args']['conversion_rate']=conversion_rate
    tool_message2=convert.invoke(input=tool_call['args'])
    print(tool_message2)
    messages.append(tool_message2)


{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1789171201, 'time_last_update_utc': 'Sat, 12 Sep 2026 00:00:01 +0000', 'time_next_update_unix': 1789257601, 'time_next_update_utc': 'Sun, 13 Sep 2026 00:00:01 +0000', 'base_code': 'USD', 'target_code': 'INR', 'conversion_rate': 95.6045}
95.6045


In [82]:
messages

[HumanMessage(content='what is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 {'result': 'success',
  'documentation': 'https://www.exchangerate-api.com/docs',
  'terms_of_use': 'https://www.exchangerate-api.com/terms',
  'time_last_update_unix': 1789171201,
  'time_last_update_utc': 'Sat, 12 Sep 2026 00:00:01 +0000',
  'time_next_update_unix': 1789257601,
  'time_next_update_utc': 'Sun, 13 Sep 2026 00:00:01 +0000',
  'base_code': 'USD',
  'target_code': 'INR',
  'conversion_rate': 95.6045}]